In [51]:
import dotenv
import os

dotenv.load_dotenv()

True

In [52]:
from importlib import metadata
import pandas as pd
from langchain_core.documents import Document

data = pd.read_csv("dataset/Tamil_movies_dataset.csv")

def generate_movie_profile(row):
    return (
        f"Title: {row['MovieName']}\n"
        f"Genre: {row['Genre']}\n"
        f"Director: {row['Director']}\n"
        f"Actor: {row['Actor']}\n"
        f"Release Year: {row['Year']}\n"
        f"Rating: {row['Rating']}\n"
    )

data["combinedrating"]=data.apply(generate_movie_profile, axis=1)

documents = [
    Document(
        page_content=row["combinedrating"],
        metadata={"source": f"Movie: {row['MovieName']}", "genre": row["Genre"], "director": row["Director"], "actor": row["Actor"], "year": row["Year"], "rating": row["Rating"]},
    )for _, row in data.iterrows()
]




In [80]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore=Chroma.from_documents(documents,embeddings,persist_directory="dataset/Tamil_movies_dataset_chroma")

In [53]:
from pydantic import BaseModel,Field
from typing import List

class Shot(BaseModel):
    shotnumber:int
    visual:str=Field(description="A brief description of the visual content of the shot.")
    cameraangle:str=Field(description="The camera angle used in the shot, e.g., close-up, wide shot, aerial view.")
    audio_cue:str=Field(description="Any significant audio cues present in the shot, such as dialogue, sound effects, or music.")

class TrailerPackage(BaseModel):
    structure: str = Field(description="The 3-act breakdown of the trailer")
    voice_over: str = Field(description="The script for the narrator it should be in tamil and that tamil is not pure it should be a mix of tamil and english in a way that it is used in common")
    music_mood: str = Field(description="Instrumentation, tempo, and vibe")
    fonrstyle: str = Field(description="The font style to be used in the trailer")
    title: str = Field(description="The title of the movie in tamil and english that is good make it catchy and appealing")
    shot_list: List[Shot]


In [54]:
from langchain_core.prompts import ChatPromptTemplate

prompt =ChatPromptTemplate.from_template(
    """
    You are a Kollywood Trailer Editor who is an expert in creating engaging and captivating trailers for Tamil movies.
    Your task is to analyze the provided movie plot and generate a detailed trailer structure that includes a 3-act breakdown, voice-over script, music mood, and a shot list with descriptions of visuals, camera angles, and audio cues using the synopsis of the movie
    MOVIE SYNOPSIS: {synopsis}
    ADDITIONAL STYLE GUIDELINES: {instructions}
    """
)

In [62]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
model=ChatGoogleGenerativeAI(model="gemini-2.5-flash",google_api_key=os.getenv("GEMINI_API_KEY"),temperature=0.7)
model1=ChatGroq(model="llama-3.3-70b-versatile",temperature=0.7,groq_api_key=os.getenv("GROQ_API_KEY"))

def generate_trailer_package(payload: dict) -> TrailerPackage:
    structeredllm=model.with_structured_output(TrailerPackage)
    synopsis = payload.get("synopsis")
    user_instructions = payload.get("user_instructions", None)
    chain=prompt|structeredllm
    response= chain.invoke({"synopsis": synopsis,"instructions": user_instructions if user_instructions else "No specific style requested."})
    data = response.model_dump()
    
    for key, value in data.items():
        title = key.replace("_", " ").upper()
        print(f"{title}")
        if isinstance(value, list):
            for item in value:
                shot_num = item.get('shot_number', '-')
                print(f"Shot {shot_num}: {item.get('visual')}")
                print(f"Camera Angle: {item.get('cameraangle')}")
                print(f"Audio Cue: {item.get('audio_cue')}\n")
        else:
            print(f"{value}\n")
            
    return response


def generate_trailer_package1(payload: dict) -> TrailerPackage:
    structeredllm=model1.with_structured_output(TrailerPackage)
    synopsis = payload.get("synopsis")
    user_instructions = payload.get("user_instructions", None)
    chain=prompt|structeredllm
    response= chain.invoke({"synopsis": synopsis,"instructions": user_instructions if user_instructions else "No specific style requested."})
    data = response.model_dump()
    
    for key, value in data.items():
        title = key.replace("_", " ").upper()
        print(f"{title}")
        if isinstance(value, list):
            for item in value:
                shot_num = item.get('shot_number', '-')
                print(f"Shot {shot_num}: {item.get('visual')}")
                print(f"Camera Angle: {item.get('cameraangle')}")
                print(f"Audio Cue: {item.get('audio_cue')}\n")
        else:
            print(f"{value}\n")
            
    return response

   

    

In [56]:
repsone=generate_trailer_package({"synopsis":"""
A fearless officer investigates a string of brutal murders, only to uncover a chilling motive behind the serial killer’s deadly game.
""","user_instructions":"Make it in lokesh kangaraj style"})  

STRUCTURE
ACT 1: The Intrigue – Establish the brutal serial murders, the fear gripping the city, and introduce the fearless officer assigned to the case. Build initial suspense and mystery around the killer's identity and methods. ACT 2: The Escalation – The officer delves deeper into the investigation, facing dead ends and growing danger. Hints of the killer's chilling motive begin to surface, leading to high-octane action sequences and a sense of impending doom. ACT 3: The Revelation & Confrontation – A glimpse of the elusive killer, a major twist revealing the depth of their madness, and the officer's relentless pursuit culminating in a dramatic standoff. Leave the audience with a powerful hook, emphasizing the personal stakes.

VOICE OVER
உலகம் முழுக்க, ஒரு பயம் பரவி இருக்கு. யாருக்கும் புரியாத murders... (A fear has spread across the world. Unexplained murders...) ஒரு fearless officer, இந்த puzzle-a solve panna poran. (A fearless officer is going to solve this puzzle.) But, the ki

In [63]:
repsone=generate_trailer_package1({"synopsis":"""
A fearless officer investigates a string of brutal murders, only to uncover a chilling motive behind the serial killer’s deadly game.
""","user_instructions":"Make it in lokesh kangaraj style"}) 

STRUCTURE
3-act breakdown: introduction to the murders, the investigation unfolds, and the cat-and-mouse chase

VOICE OVER
இது ஒரு கொலைகாரனின் விளையாட்டு, ஆனால் நீதி எப்போதும் வெல்லும், justice will always prevail

MUSIC MOOD
dark and ominous

FONRSTYLE
bold and gritty

TITLE
Vendetta: Game of Death

SHOT LIST
Shot -: close-up of a bloody crime scene
Camera Angle: low-angle shot
Audio Cue: sound of police sirens

Shot -: fearless officer walking towards the camera
Camera Angle: wide shot
Audio Cue: sound of footsteps and background music

Shot -: serial killer’s point of view, with a blurred face
Camera Angle: point-of-view shot
Audio Cue: sound of heavy breathing

Shot -: investigation scenes, with clues and suspects
Camera Angle: medium shot
Audio Cue: sound of typing on a computer and background music

Shot -: intense action sequences, with the officer chasing the killer
Camera Angle: high-angle shot
Audio Cue: sound of gunshots and explosions

